# Video Diffusion Model — Kaggle Training

Trains the custom 3D U-Net video diffusion model from scratch on Kaggle's free T4/P100 GPU (16 GB).  
No pretrained weights, no external dataset — everything is self-contained.

**Before running:**
1. Enable GPU: *Settings → Accelerator → GPU T4 x2* (or P100)
2. Enable Internet: *Settings → Internet → On*  (needed for the git clone step)
3. Set `HF_TOKEN` secret in *Settings → Secrets* if you want HuggingFace checkpoint sync
4. Set `GITHUB_REPO` below to your fork URL

In [ ]:
# ── 0. Config ─────────────────────────────────────────────────────────────────
GITHUB_REPO   = "https://github.com/YOUR_USERNAME/Video-Model.git"  # change this
REPO_DIR      = "/kaggle/working/Video-Model"
CONFIG        = "configs/train_kaggle.yaml"
HF_REPO_ID    = ""        # e.g. "your-hf-username/vdm-kaggle"  — leave blank to skip sync
SMOKE_TEST    = False     # set True to do a quick 200-step run first to verify everything works
RESUME_CKPT   = ""        # path to last.pt if resuming — leave blank for fresh start

In [ ]:
# ── 1. System info ────────────────────────────────────────────────────────────
import subprocess, sys

def run(cmd, **kw):
    print(f"$ {cmd}")
    result = subprocess.run(cmd, shell=True, check=True, text=True, capture_output=True, **kw)
    if result.stdout: print(result.stdout)
    if result.stderr: print(result.stderr)
    return result

run("nvidia-smi")
run("python --version")
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ── 2. Install dependencies ───────────────────────────────────────────────────
# Kaggle already has torch; we only need the lightweight extras.
run("pip install -q imageio imageio-ffmpeg opencv-python-headless tqdm pyyaml huggingface_hub")

In [ ]:
# ── 3. Clone repo ─────────────────────────────────────────────────────────────
import os

if os.path.isdir(REPO_DIR):
    run(f"git -C {REPO_DIR} pull")
else:
    run(f"git clone {GITHUB_REPO} {REPO_DIR}")

os.chdir(REPO_DIR)
print(f"Working dir: {os.getcwd()}")
run("ls -la")

In [ ]:
# ── 4. Smoke test (optional — 200 steps, ~3 min) ─────────────────────────────
if SMOKE_TEST:
    print("Running smoke test (200 steps)...")
    run(f"python scripts/train.py --config {CONFIG} --steps 200")
    print("Smoke test passed!")
else:
    print("Skipping smoke test (set SMOKE_TEST=True to enable)")

In [ ]:
# ── 5. Full training run ──────────────────────────────────────────────────────
# Checkpoints are saved every 2000 steps to /kaggle/working/runs/kaggle/
# They persist as Kaggle output even if the session ends.

resume_flag = f"--resume {RESUME_CKPT}" if RESUME_CKPT else ""

train_cmd = f"python scripts/train.py --config {CONFIG} {resume_flag}"
print(f"Starting training: {train_cmd}")
print("Checkpoints saved to: /kaggle/working/runs/kaggle/")

# Use subprocess without capture_output so we see live progress in the notebook
import subprocess
proc = subprocess.run(train_cmd, shell=True, cwd=REPO_DIR)
if proc.returncode != 0:
    print(f"Training exited with code {proc.returncode}")
else:
    print("Training complete!")

In [ ]:
# ── 6. Generate sample videos from the trained checkpoint ────────────────────
ckpt_path = "/kaggle/working/runs/kaggle/last.pt"
out_path  = "/kaggle/working/sample_final.gif"

if os.path.exists(ckpt_path):
    run(f"python scripts/sample.py --ckpt {ckpt_path} --n 4 --steps 100 --out {out_path}")
    print(f"Sample saved: {out_path}")
    
    # Show in notebook
    from IPython.display import Image, display
    display(Image(filename=out_path))
else:
    print(f"Checkpoint not found at {ckpt_path}")

In [ ]:
# ── 7. (Optional) Sync checkpoint to HuggingFace Hub ─────────────────────────
# This lets you download/resume the checkpoint in future sessions.
# Requires: HF_TOKEN secret set in Kaggle settings, and HF_REPO_ID filled in above.

if HF_REPO_ID:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import HfApi
    
    try:
        token = UserSecretsClient().get_secret("HF_TOKEN")
        api = HfApi(token=token)
        
        # Create repo if it doesn't exist
        api.create_repo(repo_id=HF_REPO_ID, repo_type="model", exist_ok=True)
        
        # Upload checkpoint folder
        api.upload_folder(
            folder_path="/kaggle/working/runs/kaggle",
            repo_id=HF_REPO_ID,
            repo_type="model",
            commit_message="Kaggle training checkpoint"
        )
        print(f"Checkpoint synced to: https://huggingface.co/{HF_REPO_ID}")
    except Exception as e:
        print(f"HF sync skipped: {e}")
        print("Make sure HF_TOKEN is set in Kaggle Secrets and HF_REPO_ID is correct.")
else:
    print("HF_REPO_ID not set — skipping sync. Checkpoint is in /kaggle/working/runs/kaggle/")

In [ ]:
# ── 8. List all saved files ───────────────────────────────────────────────────
import os
out_dir = "/kaggle/working/runs/kaggle"
if os.path.isdir(out_dir):
    for f in sorted(os.listdir(out_dir)):
        size = os.path.getsize(os.path.join(out_dir, f)) / 1e6
        print(f"  {f:40s}  {size:.1f} MB")
else:
    print("No output directory found yet.")